In [ ]:
# Inventory Checker - Google Colab
# Paste each "CELL" block into its own Colab cell and run top to bottom.

# ===================== CELL 1: Upload the CSV =====================
import pandas as pd
import numpy as np

try:
    from google.colab import files
    uploaded = files.upload()              # pick inventory.csv
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    CSV_PATH = "inventory project.csv"             # running outside Colab


In [ ]:

# ===================== CELL 2: Load the data =====================
REQUIRED_COLS = ["product_id", "product_name", "category", "supplier",
                 "quantity", "reorder_point", "unit_cost", "supplier_email"]

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing columns: {missing_cols}")

# Trim whitespace in text columns
for col in ["product_id", "product_name", "category", "supplier", "supplier_email"]:
    df[col] = df[col].astype("string").str.strip()

# Force numeric columns to numbers (bad values like "abc" or "$5" become NaN)
for col in ["quantity", "reorder_point", "unit_cost"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[$,]", "", regex=True),
                            errors="coerce")

print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head()



In [ ]:
# ===================== CELL 3: Validation checks =====================
issues = []  # one row per problem found

def log_issue(mask, issue, column):
    for idx in df[mask].index:
        issues.append({
            "row": idx + 2,  # +2 = header row + 1-based Excel numbering
            "product_id": df.at[idx, "product_id"],
            "column": column,
            "issue": issue,
            "value": df.at[idx, column],
        })

# 1. Missing values
for col in REQUIRED_COLS:
    log_issue(df[col].isna() | (df[col].astype(str).str.strip() == ""), "Missing value", col)

# 2. Duplicate product IDs (flags every copy, not just the 2nd one)
log_issue(df["product_id"].notna() & df["product_id"].duplicated(keep=False),
          "Duplicate product ID", "product_id")

# 3. Negative numbers
for col in ["quantity", "reorder_point", "unit_cost"]:
    log_issue(df[col] < 0, "Negative value", col)

# 4. Bonus: invalid email format
email_ok = df["supplier_email"].str.match(r"^[\w.+-]+@[\w-]+\.[\w.-]+$", na=False)
log_issue(df["supplier_email"].notna() & ~email_ok, "Invalid email format", "supplier_email")

issues_df = pd.DataFrame(issues, columns=["row", "product_id", "column", "issue", "value"])
print(f"Validation found {len(issues_df)} issue(s)")
if not issues_df.empty:
    print(issues_df["issue"].value_counts().to_string())
issues_df



In [ ]:
# ===================== CELL 4: Inventory value & reorder status =====================
df["has_data_issue"] = df.index.isin([r - 2 for r in issues_df["row"]])

# Value only counts valid, non-negative numbers
valid_qty = df["quantity"].where(df["quantity"] >= 0)
valid_cost = df["unit_cost"].where(df["unit_cost"] >= 0)
df["inventory_value"] = (valid_qty * valid_cost).round(2)

conditions = [
    df["quantity"].isna() | df["reorder_point"].isna() | (df["quantity"] < 0),
    df["quantity"] == 0,
    df["quantity"] <= df["reorder_point"],
]
choices = ["Check Data", "Out of Stock", "Low Stock"]
df["reorder_status"] = np.select(conditions, choices, default="In Stock")

# Suggested order: bring stock back up to 2x reorder point
df["suggested_order_qty"] = np.where(
    df["reorder_status"].isin(["Out of Stock", "Low Stock"]),
    (df["reorder_point"] * 2 - df["quantity"]).clip(lower=0),
    0,
).astype(int)
df["reorder_cost"] = (df["suggested_order_qty"] * valid_cost).round(2)

print("Reorder status counts:")
print(df["reorder_status"].value_counts().to_string())
print(f"\nTotal inventory value: ${df['inventory_value'].sum():,.2f}")
print(f"Estimated reorder cost: ${df['reorder_cost'].sum():,.2f}")


In [ ]:

# ===================== CELL 5: Summaries =====================
category_summary = (df.groupby("category", dropna=False)
    .agg(products=("product_id", "count"),
         total_units=("quantity", "sum"),
         inventory_value=("inventory_value", "sum"),
         low_stock=("reorder_status", lambda s: (s == "Low Stock").sum()),
         out_of_stock=("reorder_status", lambda s: (s == "Out of Stock").sum()))
    .sort_values("inventory_value", ascending=False)
    .reset_index())

reorder_list = (df[df["reorder_status"].isin(["Out of Stock", "Low Stock"])]
    [["product_id", "product_name", "supplier", "supplier_email", "quantity",
      "reorder_point", "suggested_order_qty", "unit_cost", "reorder_cost", "reorder_status"]]
    .sort_values(["reorder_status", "supplier"], ascending=[False, True]))

overview = pd.DataFrame({
    "metric": ["Total products", "Rows with data issues", "Total issues",
               "In Stock", "Low Stock", "Out of Stock", "Check Data",
               "Total inventory value ($)", "Estimated reorder cost ($)"],
    "value": [len(df), int(df["has_data_issue"].sum()), len(issues_df),
              int((df["reorder_status"] == "In Stock").sum()),
              int((df["reorder_status"] == "Low Stock").sum()),
              int((df["reorder_status"] == "Out of Stock").sum()),
              int((df["reorder_status"] == "Check Data").sum()),
              round(df["inventory_value"].sum(), 2),
              round(df["reorder_cost"].sum(), 2)],
})
display(overview) if "display" in globals() else print(overview)
category_summary


In [ ]:
# ===================== CELL 6: Export the report =====================
REPORT_XLSX = "inventory_report.xlsx"
with pd.ExcelWriter(REPORT_XLSX, engine="openpyxl") as writer:
    overview.to_excel(writer, sheet_name="Overview", index=False)
    df.to_excel(writer, sheet_name="Processed Inventory", index=False)
    reorder_list.to_excel(writer, sheet_name="Reorder List", index=False)
    category_summary.to_excel(writer, sheet_name="By Category", index=False)
    issues_df.to_excel(writer, sheet_name="Data Issues", index=False)

df.to_csv("processed_inventory.csv", index=False)
print(f"Saved {REPORT_XLSX} and processed_inventory.csv")

try:
    files.download(REPORT_XLSX)
    files.download("processed_inventory.csv")
except NameError:
    pass

In [ ]:
# ===================== CELL 7: Export to Google Sheets with alert tracking =====================
# Replaces the old Excel CELL 7. Run after CELLS 1-6 (uses the `df` DataFrame).
#
# Alert rules:
#   alert_required = TRUE when stock_status is "Low Stock" or "Out of Stock"
#   alert_sent     = stays FALSE until your email step (n8n)
#                    actually sends the email and writes TRUE back to the sheet.
#                    This script never sets it to TRUE.
#   last_alerted   = date/time of the last email, written by the email step.
#   Reset          = when a product is back to normal stock, alert_sent goes back to FALSE
#                    so the next shortage triggers a new email.

import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

SHEET_NAME = "Inventory Tracker"      # Google Sheet file name in your Drive
TAB_NAME = "Inventory"                # tab inside the sheet
TIMEZONE = "America/New_York"         # Colab runs in UTC, so set your local time
CLEAR_LAST_ALERTED_ON_RESET = False   # True = also blank out last_alerted when stock is back to normal

def to_bool(v):
    """Sheets can hand back TRUE as True, 'TRUE', or 'true' - treat them all the same."""
    return str(v).strip().upper() == "TRUE"

def build_sheet_df(df):
    """Pick the columns for the sheet and calculate stock_status / alert_required."""
    out = df[[
        "product_id", "product_name", "category", "supplier", "supplier_email",
        "quantity", "reorder_point", "unit_cost", "inventory_value",
        "reorder_status", "suggested_order_qty", "reorder_cost", "has_data_issue",
    ]].rename(columns={"reorder_status": "stock_status"}).copy()
    out["alert_required"] = out["stock_status"].isin(["Low Stock", "Out of Stock"])
    out["last_checked"] = datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y-%m-%d %H:%M:%S")
    return out

def merge_alert_state(new_df, old_df):
    """Carry alert_sent / last_alerted over from the existing sheet, then apply the reset rule."""
    new_df = new_df.copy()
    if old_df is not None and not old_df.empty and "product_id" in old_df.columns:
        old = old_df.copy()
        old["product_id"] = old["product_id"].astype(str).str.strip()
        old = old.drop_duplicates("product_id", keep="first").set_index("product_id")
        ids = new_df["product_id"].astype(str)
        prev_sent = ids.map(old["alert_sent"]) if "alert_sent" in old else pd.Series(index=new_df.index, dtype=object)
        prev_time = ids.map(old["last_alerted"]) if "last_alerted" in old else pd.Series(index=new_df.index, dtype=object)
        new_df["alert_sent"] = prev_sent.apply(to_bool)
        new_df["last_alerted"] = prev_time.fillna("").astype(str).replace({"nan": "", "None": ""})
    else:
        # First run (or brand-new products): nothing has been emailed yet
        new_df["alert_sent"] = False
        new_df["last_alerted"] = ""

    # Reset rule: product is back to normal -> ready for a future alert
    back_to_normal = ~new_df["alert_required"]
    new_df.loc[back_to_normal, "alert_sent"] = False
    if CLEAR_LAST_ALERTED_ON_RESET:
        new_df.loc[back_to_normal, "last_alerted"] = ""
    return new_df

def to_sheet_values(frame):
    """Turn the DataFrame into plain rows Google Sheets accepts (no NaN, no numpy types)."""
    def clean(v):
        if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA:
            return ""
        if hasattr(v, "item"):          # numpy number/bool -> plain Python
            v = v.item()
        if isinstance(v, bool):
            return "TRUE" if v else "FALSE"
        return v
    return [list(frame.columns)] + [[clean(v) for v in row] for row in frame.itertuples(index=False)]

# ---------------- Connect to Google Sheets (Colab) or use a local file (testing) ----------------
sheet_df = build_sheet_df(df)

try:
    from google.colab import auth
    import google.auth, gspread
    auth.authenticate_user()                       # pop-up: sign in with your Google account
    creds, _ = google.auth.default()
    gc = gspread.authorize(creds)

    try:
        sh = gc.open(SHEET_NAME)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)
        print(f"Created new Google Sheet: {SHEET_NAME}")
    try:
        ws = sh.worksheet(TAB_NAME)
        old_df = pd.DataFrame(ws.get_all_records())
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=TAB_NAME, rows=len(sheet_df) + 50, cols=len(sheet_df.columns) + 5)
        old_df = None

    final_df = merge_alert_state(sheet_df, old_df)
    ws.clear()
    ws.update(range_name="A1", values=to_sheet_values(final_df), value_input_option="USER_ENTERED")
    ws.freeze(rows=1)
    print(f"Updated '{TAB_NAME}' in '{SHEET_NAME}': {sh.url}")

except ImportError:
    # Not in Colab: same logic against a local CSV so you can test it
    LOCAL = "inventory_sheet.csv"
    try:
        old_df = pd.read_csv(LOCAL, dtype=str, keep_default_na=False)
    except FileNotFoundError:
        old_df = None
    final_df = merge_alert_state(sheet_df, old_df)
    pd.DataFrame(to_sheet_values(final_df)[1:], columns=final_df.columns).to_csv(LOCAL, index=False)
    print(f"Not in Colab - saved to {LOCAL}")

# ---------------- Confirm ----------------
for col in ["stock_status", "reorder_point", "alert_required", "alert_sent", "last_alerted"]:
    assert col in final_df.columns, f"Missing column: {col}"
waiting = final_df["alert_required"] & ~final_df["alert_sent"]
print(f"{len(final_df)} products | alert_required: {int(final_df['alert_required'].sum())} "
      f"| alert_sent: {int(final_df['alert_sent'].sum())} | waiting for email: {int(waiting.sum())}")
final_df[["product_id", "product_name", "quantity", "reorder_point",
          "stock_status", "alert_required", "alert_sent", "last_alerted"]].head(10)